# Domain Pretraining of TinyLlama with LoRA
This notebook fine-tunes **TinyLlama-1.1B** on domain text extracted from a PDF, using LoRA for parameter-efficient training.

**Sections:**
1. Install dependencies
2. Extract & prepare data from PDF
3. Tokenize the dataset
4. Load model & apply LoRA
5. Train

In [2]:
!pip uninstall -y torch torchvision torchaudio torchao transformers accelerate peft bitsandbytes -q

!pip install -q \
    "torch==2.5.1" \
    "torchvision==0.20.1" \
    "torchaudio==2.5.1" \
    "transformers==4.48.3" \
    "accelerate==1.2.1" \
    "peft==0.14.0" \
    "bitsandbytes==0.45.0" \
    "datasets==3.2.0" \
    PyMuPDF

!pip check

bigframes 2.39.0 requires google-cloud-bigquery-storage, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage, which is not installed.
tpot 1.1.0 has requirement dill>=0.3.9, but you have dill 0.3.8.
google-colab 1.0.0 has requirement jupyter-server==2.14.0, but you have jupyter-server 2.12.5.
google-colab 1.0.0 has requirement pandas==2.2.2, but you have pandas 2.3.3.
dopamine-rl 4.1.2 has requirement gym<=0.25.2, but you have gym 0.26.2.
moviepy 1.0.3 has requirement decorator<5.0,>=4.0.2, but you have decorator 5.3.1.
gcsfs 2025.3.0 has requirement fsspec==2025.3.0, but you have fsspec 2024.9.0.


> After the install finishes, **restart the kernel** now, then run the cells below from the top.

## 2. Extract & prepare data from PDF
Set `PDF_PATH` to your source PDF, then extract text and split it into training-sized paragraph chunks.

In [3]:
import re
import fitz  # PyMuPDF
from datasets import Dataset

# Update this to your PDF's location
PDF_PATH = "/kaggle/input/datasets/homeeedits/ml-text-book/2af5365a3f0d24cc2ee9f787bbab14e9_MIT18_409S15_bookex.pdf"

In [4]:
def extract_text_from_pdf(pdf_path):
    """Return a list of page-level text strings from a PDF."""
    text_blocks = []
    
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks


def split_paragraphs(pages, min_words=30, max_words=200):
    """Split page text into paragraph chunks within a word-count range."""
    chunks = []
    for page_text in pages:
        for paragraph in re.split(r"\n\s*\n", page_text):
            words = paragraph.strip().split()
            if len(words) < min_words:
                continue
            chunks.append(" ".join(words[:max_words]))
    return chunks

In [6]:
pdf_texts = extract_text_from_pdf(PDF_PATH)
paragraphs = split_paragraphs(pdf_texts)

dataset = Dataset.from_list([{"text": p} for p in paragraphs])
print(f"{len(dataset)} training examples")
dataset[0]

147 training examples


{'text': 'Algorithmic Aspects of Machine Learning Ankur Moitra c© Draft date March 30, 2014 Algorithmic Aspects of Machine Learning ©2015 by Ankur Moitra. Note: These are unpolished, incomplete course notes. Developed for educational use at MIT and for publication through MIT OpenCourseware.'}

## 3. Tokenize the dataset

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [8]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=384,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_dataset

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 147
})

## 4. Load model & apply LoRA

In [9]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [10]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## 5. Train

In [11]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [12]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
20,2.110200
40,0.972100


TrainOutput(global_step=45, training_loss=1.4657542864481607, metrics={'train_runtime': 467.643, 'train_samples_per_second': 1.572, 'train_steps_per_second': 0.096, 'total_flos': 1593922654568448.0, 'train_loss': 1.4657542864481607, 'epoch': 4.54054054054054})